# Notebook 04 — Paper Figures
**Normative Geometry of Language Models: A Cross-Architectural Study**

Generates all 5 paper figures. Requires notebooks 01–03 to have been run first.

- Fig 1: Cross-model Spearman ρ heatmap
- Fig 2: Latent category hierarchy
- Fig 3: Representational–behavioral dissociation scatter
- Fig 4: RLHF delta lollipop (Llama-3 vs Mistral)
- Fig 5: R_HAT geometric tension vs RLHF behavioral effect

In [ ]:
# ── CELL 0 — SETUP ────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import TwoSlopeNorm
from scipy.stats import spearmanr, linregress
from pathlib import Path

# ─── CONFIGURE THESE PATHS ────────────────────────────────────────────────────
MANIFEST_DIR  = Path("./results/manifests")
REFUSAL_DIR   = Path("./results/out_refusal")
REPENG_DIR    = Path("./results/out_repeng")
FIGURES_DIR   = Path("./results/figures")
# ─────────────────────────────────────────────────────────────────────────────

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Publication style
plt.rcParams.update({
    "font.family": "serif", "font.size": 9,
    "axes.titlesize": 10, "axes.titleweight": "bold",
    "axes.labelsize": 9, "xtick.labelsize": 8, "ytick.labelsize": 8,
    "legend.fontsize": 8, "figure.dpi": 150,
    "axes.spines.top": False, "axes.spines.right": False,
})

MODELS_META = {
    "distilgpt2":          {"family":"GPT-2",   "group":"base",     "label":"DistilGPT-2"},
    "qwen2-0.5b":          {"family":"Qwen2",   "group":"base",     "label":"Qwen2-0.5B"},
    "gemma-3-270m":        {"family":"Gemma-3", "group":"base",     "label":"Gemma-3-270M"},
    "tinyllama-1.1b":      {"family":"Llama",   "group":"chat",     "label":"TinyLlama-1.1B"},
    "phi3-mini":           {"family":"Phi-3",   "group":"instruct", "label":"Phi-3-mini"},
    "llama3-8b-base":      {"family":"Llama-3", "group":"base",     "label":"Llama-3.1-8B"},
    "llama3-8b-instruct":  {"family":"Llama-3", "group":"instruct", "label":"Llama-3.1-8B-IT"},
    "mistral-7b-base":     {"family":"Mistral", "group":"base",     "label":"Mistral-7B"},
    "mistral-7b-instruct": {"family":"Mistral", "group":"instruct", "label":"Mistral-7B-IT"},
}
models  = list(MODELS_META.keys())
labels  = [MODELS_META[m]["label"] for m in models]
BASE_COL, INST_COL, CHAT_COL = "#2c5f8a", "#c0392b", "#7f8c8d"
GROUP_COLS = {"base": BASE_COL, "instruct": INST_COL, "chat": CHAT_COL, "chat-light": CHAT_COL}

# Load all data
all_dfs = []
for name, meta in MODELS_META.items():
    f = MANIFEST_DIR / f"{name}_tensions.csv"
    if f.exists():
        df = pd.read_csv(f)
        pol = df["polarity"].iloc[0]
        t = df["tension"].values * pol
        df["tension_norm"] = (t - t.mean()) / t.std()
        df["model"] = name
        all_dfs.append(df)
all_df = pd.concat(all_dfs, ignore_index=True)

sp_df  = pd.read_csv(MANIFEST_DIR / "spearman.csv", index_col=0)
ref    = pd.read_csv(REFUSAL_DIR  / "refusal_summary.csv")
cross  = pd.read_csv(REFUSAL_DIR  / "rhat_vs_rlhf_cross.csv")
cat_stats = all_df.groupby("category")["tension_norm"].agg(["mean","std"]).sort_values("mean", ascending=False)

print(f"Data loaded: {len(all_df)} rows, {all_df['model'].nunique()} models")

In [ ]:
# ── CELL 1 — FIGURE 1: SPEARMAN HEATMAP ──────────────────────────────────────
sp = sp_df.values
n  = len(models)

fig, ax = plt.subplots(figsize=(6.5, 5.2))
norm = TwoSlopeNorm(vmin=-0.3, vcenter=0, vmax=1.0)
im   = ax.imshow(sp, cmap="RdBu_r", norm=norm)

for i in range(n):
    for j in range(n):
        v  = sp[i, j]
        tc = "white" if abs(v) > 0.6 else "#1a1a1a"
        ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                fontsize=7.5, color=tc, fontweight="bold" if i==j else "normal")

# Family separator lines
prev, breaks = MODELS_META[models[0]]["family"], []
for idx, m in enumerate(models[1:], 1):
    if MODELS_META[m]["family"] != prev:
        breaks.append(idx - 0.5)
    prev = MODELS_META[m]["family"]
for b in breaks:
    ax.axvline(b, color="#888", lw=0.6, ls="--", alpha=0.5)
    ax.axhline(b, color="#888", lw=0.6, ls="--", alpha=0.5)

ax.set_xticks(range(n)); ax.set_yticks(range(n))
ax.set_xticklabels(labels, rotation=35, ha="right", fontsize=7.5)
ax.set_yticklabels(labels, fontsize=7.5)
for tick, m in zip(ax.get_xticklabels(), models): tick.set_color(GROUP_COLS[MODELS_META[m]["group"]])
for tick, m in zip(ax.get_yticklabels(), models): tick.set_color(GROUP_COLS[MODELS_META[m]["group"]])

plt.colorbar(im, ax=ax, fraction=0.035).set_label("Spearman ρ", fontsize=8)
ax.legend(handles=[mpatches.Patch(color=c, label=l) for c, l in
          [(BASE_COL,"Base"),(INST_COL,"Instruct"),(CHAT_COL,"Chat")]],
          loc="upper left", fontsize=7.5, framealpha=0.85)
ax.set_title("Figure 1. Cross-model Spearman ρ on normalized tension profiles", fontsize=9, pad=10)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "fig1_spearman_heatmap.pdf", bbox_inches="tight")
plt.savefig(FIGURES_DIR / "fig1_spearman_heatmap.png", bbox_inches="tight", dpi=200)
plt.show()
print("✓ Figure 1 saved")

In [ ]:
# ── CELL 2 — FIGURE 2: CATEGORY HIERARCHY ────────────────────────────────────
fig, ax = plt.subplots(figsize=(6.0, 4.0))
cats, means, stds = cat_stats.index.tolist(), cat_stats["mean"].values, cat_stats["std"].values
colors = [INST_COL if m > 0 else BASE_COL for m in means]
ax.barh(range(len(cats)), means, xerr=stds, color=colors, alpha=0.80, height=0.65,
        error_kw={"elinewidth":0.8,"capsize":2.5,"ecolor":"#555"}, linewidth=0)
ax.axvline(0, color="#333", lw=0.8)
for i, (m, _) in enumerate(zip(means, stds)):
    ax.text(m + (0.05 if m>=0 else -0.05), i, f"{m:+.2f}", va="center",
            ha="left" if m>=0 else "right", fontsize=7.5)
ax.set_yticks(range(len(cats))); ax.set_yticklabels(cats, fontsize=8)
ax.set_xlabel("Mean normalized tension τ̄ (z-score ± 1 SD)", fontsize=8)
ax.set_xlim(-2.2, 2.2); ax.invert_yaxis()
ax.spines["left"].set_visible(False); ax.tick_params(left=False)
ax.legend(handles=[mpatches.Patch(color=INST_COL, alpha=0.8, label="Positive τ̄"),
                   mpatches.Patch(color=BASE_COL,  alpha=0.8, label="Negative τ̄")],
          fontsize=7.5, loc="lower right", framealpha=0.85)
ax.set_title("Figure 2. Latent category hierarchy in JBB-Behaviors", fontsize=9, pad=8)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "fig2_category_hierarchy.pdf", bbox_inches="tight")
plt.savefig(FIGURES_DIR / "fig2_category_hierarchy.png", bbox_inches="tight", dpi=200)
plt.show()
print("✓ Figure 2 saved")

In [ ]:
# ── CELLS 3–5 — FIGURES 3, 4, 5 ──────────────────────────────────────────────
# (same code as Cells 1–2 pattern above)
# See the full notebook at github.com/your-username/normative-geometry-llm
print("See full repository for Figures 3–5 code.")
print("All figures generated in:", FIGURES_DIR)